## Sistema de Recomendação de Cartões de Crédito

Este notebook tem como objetivo:
1. Desenvolver um modelo preditivo para classificação do principal cartão para clientes
2. Aplicar o modelo de clientes na base prospects
3. Gerar arquivo final com cartão ideal (recomendação) para prospects

### Importar bibliotecas necessárias

In [ ]:
# EDA e Visualização de Dados
import pandas as pd
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.graph_objects as go
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import chi2_contingency, f_oneway
from colorama import Fore, Back, Style

# Configurar formato de exibição para não usar notação científica
pd.set_option('display.float_format', lambda x: '%.5f' % x)
np.set_printoptions(suppress=True, precision=5)
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

# ML
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import classification_report, confusion_matrix, log_loss
from catboost import CatBoostClassifier, Pool, cv

# Otimização
import optuna

# Utilitários
import joblib
import math

# Abrir Base Clientes

In [ ]:
df_clientes = pd.read_csv('datasets/clientes.csv')

In [ ]:
df_clientes.info()

In [ ]:
# Remover colunas únicas
df_clientes.drop(columns=['ID_Cliente', 'Nome'], axis=1, inplace=True)

In [ ]:
num_vars = df_clientes.select_dtypes(include=['number']).columns
cat_vars = df_clientes.select_dtypes(include=['object']).columns
target = 'Principal Cartão'

# EDA

## Testes de Hipóteses

In [ ]:
# Testes de hipóteses entre Target Categórica e Numéricas (ANOVA)
for num_col in num_vars:
    groups = [df_clientes[df_clientes[target] == val][num_col] for val in df_clientes[target].unique()]
    stat, p = f_oneway(*groups)
    print(f"{Fore.RED if p < 0.05 else Fore.WHITE}"
            f"ANOVA entre {num_col} e {target}: p-valor = {p}")

## Analisando relação entre variáveis explicativas e targets

In [ ]:

for col in num_vars:
    fig = px.box(df_clientes, x=target, y=col, title=f"{col} por {target}")
    fig.show()

for col in cat_vars:
    fig = px.histogram(df_clientes, x=col, color=target, barmode='group', title=f"{col} por {target}")
    fig.show()

# Modelo Catboost com Validação Cruzada

In [ ]:
selected_features = ['Viagens', 'Restaurantes', 'Entretenimento', 'Cashback', 'Compras online', 'Farmácias',
                     'Programas de Milhagem', 'Postos de Combustível', 'Mercados', 'Score']
X = df_clientes[selected_features]
y = df_clientes[target]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.5, random_state=42)

# Definir os parâmetros do modelo
params = {
    'iterations': 1000,
    'learning_rate': 0.05,
    'depth': 6,
    'l2_leaf_reg': 3,
    'random_strength': 2,
    'loss_function': 'MultiClass',  # Use 'MultiClass' para problemas de classificação multiclasse
    'eval_metric': 'MultiClass'
}

model = CatBoostClassifier(**params, random_seed=42, auto_class_weights='Balanced')

model.fit(X_train, y_train)
# Obter o log loss médio
best_score = model.best_score_['learn']['MultiClass']
print(f"LogLoss no treinamento: {best_score}")

In [ ]:
y_pred = model.predict(X_test)
probs = model.predict_proba(X_test)
test_loss = log_loss(y_test, probs)
print(f"LogLoss no conjunto de teste: {test_loss:.2f}")

## Otimizar com Optuna

In [ ]:
# Definir a função objetivo para otimização
def objective(trial):
    # Definir o espaço de busca para os hiperparâmetros
    params = {
        'iterations': trial.suggest_int('iterations', 100, 1000),
        'depth': trial.suggest_int('depth', 4, 10),
        'learning_rate': trial.suggest_loguniform('learning_rate', 0.01, 0.1),
        'l2_leaf_reg': trial.suggest_loguniform('l2_leaf_reg', 1, 10),
        'random_strength': trial.suggest_uniform('random_strength', 0, 10),
        'loss_function': 'MultiClass',
        'eval_metric': 'MultiClass'
    }
    
    model = CatBoostClassifier(**params, random_seed=42, auto_class_weights='Balanced')

    model.fit(X_train, y_train)
    best_score = model.best_score_['learn']['MultiClass']

    return best_score

# Criar um estudo e otimizar
study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=10)  # Ajuste o número de trials conforme necessário

# Obter os melhores parâmetros
best_params = study.best_params
best_metric = study.best_value
print("Melhores parâmetros:", best_params)
print("Melhor métrica:", best_metric)

In [ ]:
# Treinar o modelo final com os parâmetros otimizados
best_model = CatBoostClassifier(**best_params,
                               verbose=False,
                               random_seed=42)
best_model.fit(X_train, y_train)

y_pred = best_model.predict(X_test)
probs = best_model.predict_proba(X_test)
test_loss = log_loss(y_test, probs)
print(f"LogLoss no conjunto de teste: {test_loss:.2f}")

In [ ]:
# Calcular e exibir métricas
print("\nRelatório de Classificação:")
print(classification_report(y_test, y_pred))

# Criar matriz de confusão
plt.figure(figsize=(10, 8))
conf_matrix = confusion_matrix(y_test, y_pred)
sns.heatmap(conf_matrix, annot=True, fmt='d', cmap='Blues')
plt.title('Matriz de Confusão')
plt.xlabel('Predito')
plt.ylabel('Real')
plt.show()

# Plotar importância das features
plt.figure(figsize=(10, 6))
feature_importance = pd.DataFrame({
    'feature': selected_features,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=True)

plt.barh(feature_importance['feature'], feature_importance['importance'])
plt.title('Importância das Features')
plt.xlabel('Importância')
plt.tight_layout()
plt.show()

## Salvar Modelo

In [ ]:
joblib.dump(best_model, 'modelo_recomendacao.pkl')